# Create a Reference H2O Binary Model

Train a small five-feature H2O GBM, save the version-specific binary model, create golden input/expected files, write checksums, and prove save/load parity. Output is written under ignored `workshop/outputs`.

> This is an H2O binary model, not a portable MOJO zip. It must be loaded with the same H2O version.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/01_create_reference_mojo.ipynb`.

In [2]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys

import h2o
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from h2o.estimators.gbm import H2OGradientBoostingEstimator

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

H2O_VERSION = os.environ["H2O_VERSION"]
FEATURES = [value.strip() for value in os.environ["H2O_FEATURES"].split(",") if value.strip()]
CATEGORICAL_FEATURES = [value.strip() for value in os.environ["H2O_CATEGORICAL_FEATURES"].split(",") if value.strip()]
TARGET = os.environ["H2O_TARGET"]
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]
DATA_PATH = WORKSHOP_ROOT / "data/taxi/raw/yellowTaxiData.csv"
OUTPUT_DIR = WORKSHOP_ROOT / "outputs/h2o_reference_bundle"
SEED = 42

if h2o.__version__ != H2O_VERSION:
    raise RuntimeError(f"Expected h2o=={H2O_VERSION}, found {h2o.__version__}")
if not shutil.which("java"):
    raise FileNotFoundError("OpenJDK 17 is required on PATH")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

taxi = pd.read_csv(DATA_PATH)
taxi["pickupHour"] = pd.to_datetime(taxi["tpepPickupDateTime"]).dt.hour
model_data = taxi.loc[
    (taxi["tripDistance"] > 0) & (taxi[TARGET] > 0),
    [*FEATURES, TARGET],
].dropna()
print(f"Training rows: {len(model_data):,}")

try:
    h2o.init(max_mem_size="2G", nthreads=-1)
    h2o_data = h2o.H2OFrame(model_data)
    for column in CATEGORICAL_FEATURES:
        h2o_data[column] = h2o_data[column].asfactor()

    train, validation = h2o_data.split_frame(ratios=[0.8], seed=SEED)
    model = H2OGradientBoostingEstimator(
        model_id="taxi-fare-gbm",
        ntrees=60,
        max_depth=5,
        learn_rate=0.08,
        seed=SEED,
    )
    model.train(x=FEATURES, y=TARGET, training_frame=train, validation_frame=validation)
    performance = model.model_performance(validation)
    print({"rmse": performance.rmse(), "mae": performance.mae(), "r2": performance.r2()})

    model_path = Path(h2o.save_model(model=model, path=str(OUTPUT_DIR), filename="taxi-fare-gbm", force=True))
    golden = validation.as_data_frame().head(20)
    golden_input_path = OUTPUT_DIR / "golden_input.csv"
    golden_expected_path = OUTPUT_DIR / "golden_expected.csv"
    golden[FEATURES].to_csv(
        golden_input_path, index=False, lineterminator="\n"
    )

    golden_frame = h2o.H2OFrame(golden[FEATURES])
    for column in CATEGORICAL_FEATURES:
        golden_frame[column] = golden_frame[column].asfactor()
    model.predict(golden_frame).as_data_frame().to_csv(
        golden_expected_path, index=False, lineterminator="\n"
    )

    def sha256(path: Path) -> str:
        return hashlib.sha256(path.read_bytes()).hexdigest()

    manifest = {
        "model_name": MODEL_NAME,
        "model_version": MODEL_VERSION,
        "model_format": "h2o_binary",
        "h2o_version": h2o.__version__,
        "model_file": model_path.name,
        "target": TARGET,
        "features": FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "files": {
            model_path.name: sha256(model_path),
            golden_input_path.name: sha256(golden_input_path),
            golden_expected_path.name: sha256(golden_expected_path),
        },
    }
    (OUTPUT_DIR / "model_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

    loaded_model = h2o.load_model(str(model_path))
    actual = loaded_model.predict(golden_frame).as_data_frame()
    expected = pd.read_csv(golden_expected_path)
    np.testing.assert_allclose(expected["predict"], actual["predict"], rtol=1e-6, atol=1e-6)
    print(f"Created and reloaded: {OUTPUT_DIR.relative_to(WORKSHOP_ROOT)}")
finally:
    if h2o.connection() is not None:
        h2o.cluster().shutdown(prompt=False)

Training rows: 4,964
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.18-internal" 2026-01-20; OpenJDK Runtime Environment (build 17.0.18-internal+0-adhoc.rattler.src); OpenJDK 64-Bit Server VM (build 17.0.18-internal+0-adhoc.rattler.src, mixed mode, sharing)
  Starting server from /anaconda/envs/azureml-workshop/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpfejzxaqh
  JVM stdout: /tmp/tmpfejzxaqh/h2o_azureuser_started_from_python.out
  JVM stderr: /tmp/tmpfejzxaqh/h2o_azureuser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 4 days
H2O_cluster_name:,H2O_from_python_azureuser_ljpqmr
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
gbm Model Build progress: |██████████████████████████████████████████████████████| (done) 100%
{'rmse': 2.9709854940143163, 'mae': 1.5734441920470132, 'r2': 0.911051504780502}
Parse progress: |

/anaconda/envs/azureml-workshop/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


████████████████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
Created and reloaded: outputs/h2o_reference_bundle
H2O session _sid_891b closed.


/anaconda/envs/azureml-workshop/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
/anaconda/envs/azureml-workshop/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


## Expected Result

An ignored `outputs/h2o_reference_bundle` directory contains the binary model, manifest, golden input, and golden expected output. Reloaded predictions match the source model.

Next: `02_validate_reference_bundle.ipynb`.